# Telco Customer Churn — End-to-End Analysis

**Author:** Naveen | **Role focus:** Data Scientist / AI-ML Analyst / Data Analyst

This notebook walks through the full workflow: business framing, data cleaning,
exploratory data analysis (EDA), feature engineering, model building, evaluation,
and business recommendations.

> **Goal:** predict which customers are likely to churn so the business can target
> retention offers *before* they leave.


## 1. Business Problem

A telecom company loses revenue every time a customer cancels (churns). Acquiring a
new customer typically costs far more than retaining an existing one, so even a small
reduction in churn has a large financial impact.

**Objective:** build a classification model that flags customers at high risk of
churning, and surface the *drivers* of churn so retention teams can act.

**Users:** retention / CRM teams, marketing, and customer-success leadership.

**Dataset:** IBM "Telco Customer Churn" — 7,043 customers, 21 columns. Each row is a
customer; the `Churn` column (Yes/No) is the target.


In [1]:
import sys, os

# Make the notebook runnable from either the project root or the notebooks/ folder.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_preprocessing import load_data, clean_data, get_feature_target
from feature_engineering import add_engineered_features
from model_training import train_and_compare, save_artifacts, best_model, text_report
import visualization as viz

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## 2. Load the Data

In [2]:
raw = load_data("data/raw/Telco-Customer-Churn.csv")
print("Shape:", raw.shape)
raw.head()


Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

### Data quality check

`TotalCharges` is stored as text and has 11 blank values for brand-new customers
(tenure = 0). We convert it to numeric and impute those blanks with 0 (they have
not been billed yet).


In [4]:
# Count blank TotalCharges before cleaning
blanks = pd.to_numeric(raw["TotalCharges"], errors="coerce").isna().sum()
print("Blank TotalCharges values:", blanks)
raw.loc[pd.to_numeric(raw["TotalCharges"], errors="coerce").isna(), ["tenure", "TotalCharges"]].head()


Blank TotalCharges values: 11


,tenure,TotalCharges
488,0,
753,0,
936,0,
1082,0,
1340,0,


## 3. Clean the Data

In [5]:
clean = clean_data(raw)
print("Churn rate:", round(clean["Churn"].mean(), 4))
print("Missing values remaining:", int(clean.isna().sum().sum()))
clean.head()


Churn rate: 0.2654
Missing values remaining: 0


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## 4. Exploratory Data Analysis

We look at class balance, churn by contract type, tenure distribution, and numeric
correlations. Figures are also saved to `reports/figures/`.


In [6]:
viz.plot_churn_balance(clean)
viz.plot_churn_by_contract(clean)
viz.plot_tenure_distribution(clean)
viz.plot_correlation_heatmap(clean)
print("EDA figures saved to reports/figures/")


EDA figures saved to reports/figures/


In [7]:
# Churn rate by contract type (numeric view)
clean.groupby("Contract")["Churn"].agg(["mean", "count"]).round(3)


,mean,count
Contract,,
Month-to-month,0.427,3875
One year,0.113,1473
Two year,0.028,1695


In [8]:
# Churn rate by some key categorical drivers
for col in ["InternetService", "PaymentMethod", "TechSupport"]:
    print(f"\n--- Churn rate by {col} ---")
    print(clean.groupby(col)["Churn"].mean().round(3).sort_values(ascending=False))



--- Churn rate by InternetService ---
InternetService
Fiber optic    0.419
DSL            0.190
No             0.074
Name: Churn, dtype: float64

--- Churn rate by PaymentMethod ---
PaymentMethod
Electronic check             0.453
Mailed check                 0.191
Bank transfer (automatic)    0.167
Credit card (automatic)      0.152
Name: Churn, dtype: float64

--- Churn rate by TechSupport ---
TechSupport
No                     0.416
Yes                    0.152
No internet service    0.074
Name: Churn, dtype: float64


### EDA takeaways
- Overall churn rate is ~26.5% — imbalanced, so we evaluate with ROC-AUC, precision,
  recall and F1, not just accuracy.
- Month-to-month contracts churn far more than one/two-year contracts.
- Fiber-optic internet, electronic-check payment, and short tenure are associated
  with higher churn.


## 5. Feature Engineering

We add three interpretable features:
- `tenure_group` — binned customer lifetime.
- `avg_monthly_spend` — TotalCharges / tenure (normalized spend).
- `has_streaming` — whether the customer uses any streaming service.

Categorical encoding and scaling are handled inside a leak-free
`ColumnTransformer` (fit on training folds only).


In [9]:
X, y = get_feature_target(clean)
X = add_engineered_features(X)
print("Feature matrix shape:", X.shape)
X[["tenure", "tenure_group", "avg_monthly_spend", "has_streaming"]].head()


Feature matrix shape: (7043, 22)


,tenure,tenure_group,avg_monthly_spend,has_streaming
0,1,0-12,29.85,0
1,34,24-48,55.57,0
2,2,0-12,54.08,0
3,45,24-48,40.91,0
4,2,0-12,75.82,0


## 6. Model Building & Comparison

We compare three classifiers inside a single pipeline:
- Logistic Regression (interpretable baseline)
- Random Forest
- Gradient Boosting

Class weights are balanced where supported, and the split is stratified.


In [10]:
results, fitted, (X_train, X_test, y_train, y_test) = train_and_compare(X, y)
results_df = pd.DataFrame([r.__dict__ for r in results]).set_index("name")
results_df


,accuracy,precision,recall,f1,roc_auc
name,,,,,
Logistic Regression,0.7381,0.5043,0.7861,0.6144,0.8424
Random Forest,0.7622,0.5452,0.6283,0.5839,0.8245
Gradient Boosting,0.7970,0.6560,0.4947,0.5640,0.8428


## 7. Best Model Evaluation

In [11]:
best_name = best_model(results)
print("Best model (by ROC-AUC):", best_name)
best_pipe = fitted[best_name]

viz.plot_confusion_matrix(best_pipe, X_test, y_test, best_name)
viz.plot_roc_curve(best_pipe, X_test, y_test, best_name)
viz.plot_feature_importance(best_pipe, best_name)

print(text_report(best_pipe, X_test, y_test))


Best model (by ROC-AUC): Gradient Boosting


              precision    recall  f1-score   support

        Stay       0.83      0.91      0.87      1035
       Churn       0.66      0.49      0.56       374

    accuracy                           0.80      1409
   macro avg       0.74      0.70      0.72      1409
weighted avg       0.79      0.80      0.79      1409

Confusion matrix [rows=actual, cols=pred]:
[[938  97]
 [189 185]]


## 8. Save Artifacts

In [12]:
best_saved, payload = save_artifacts(results, fitted)
print("Saved best model:", best_saved)
print("Metrics written to reports/metrics.json")


Saved best model: Gradient Boosting
Metrics written to reports/metrics.json


## 9. Business Interpretation & Recommendations

**Key churn drivers (from feature importance):**
- **Short tenure** — new customers are the most fragile; the first year is critical.
- **Month-to-month contracts** — by far the highest-churn segment.
- **Fiber-optic internet** and **electronic-check payments** correlate with churn,
  likely reflecting price sensitivity and friction.
- **Lack of tech support / online security** add-ons increases churn risk.

**Recommended actions:**
1. Offer incentives to convert month-to-month customers to annual contracts.
2. Run a targeted onboarding / early-life retention program in the first 12 months.
3. Promote auto-pay (card/bank) over electronic check to reduce payment friction.
4. Bundle tech support / security add-ons, especially for fiber customers.

**How the model is used:** score the active customer base monthly, rank by churn
probability, and route the top-risk customers to retention campaigns. This converts a
predictive score into a concrete, prioritized action list.

## 10. Assumptions & Limitations
- The dataset is a fictional IBM sample; absolute numbers will differ from any real
  company, but the *workflow* and *relationships* are realistic.
- 11 tenure-0 customers had blank `TotalCharges`, imputed with 0.
- Models use default-ish hyperparameters; a production version would add
  cross-validated tuning and threshold optimization for the business cost of
  false negatives vs false positives.

## 11. Future Improvements
- Hyperparameter tuning (GridSearchCV / Optuna) and probability-threshold tuning.
- Try XGBoost / LightGBM and SHAP for per-customer explanations.
- Deploy as a REST API (FastAPI) or batch scoring job; add monitoring for drift.
